### https://www.kaggle.com/datasets/rgupt44/wealth-management-customer-data

In [34]:
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

### 1. import data

In [35]:
df = pd.read_csv('Bank_Customers.csv')
df['LastTransactionDate'] = pd.to_datetime(df['LastTransactionDate'])
df['DaysSinceLastTransaction'] = (df['LastTransactionDate'].max() - df['LastTransactionDate']).dt.days

df = df.drop(['RowNumber', 'Surname', 'Retention', 'CLV', 'LastTransactionDate'], axis=1)
df = df.dropna(axis=0)
df.shape

(10000, 35)

### 2. randomly make missing

In [36]:
np.random.seed(42)
for i in df.columns[1:-3]:
    missing_rate = np.random.uniform(0.05, 0.12)
    n_missing = int(len(df) * missing_rate)
    
    missing_indices = np.random.choice(df.index, size=n_missing, replace=False)
    df.loc[missing_indices, i] = np.nan
print(df.shape)
print(df.isna().sum())

(10000, 35)
CustomerID                     0
CreditScore                  762
Country                     1045
Gender                      1032
Married                      915
Age                         1108
Dependents                   622
NumBankAccts                 744
HasCrCard                   1044
EmergingMarketFund           889
RealEstate                  1050
PrivateEquity                763
GovtBonds                   1121
CorpBonds                    516
ETF Tech                     922
ETF Health                   676
ETF Med                      926
EstimatedSalary              526
Mortgage                     702
Risk Profile                 548
Debt                         569
Net Assets                  1053
Portfolio Return             598
Diversification              681
BusinessOwner                639
Revenue                      684
Margin                       734
LifeInsurance                945
NumTransactions              516
LastTransactionAmt           67

### 3. export data

In [37]:
df = df[['CustomerID', 'Gender', 'Married', 'Age', 'CreditScore', 'Dependents', 'NumBankAccts', 'HasCrCard', 
         'EmergingMarketFund', 'RealEstate', 'PrivateEquity', 'GovtBonds', 'CorpBonds', 'ETF Tech', 'ETF Health', 
         'ETF Med', 'Debt', 'Net Assets', 'Mortgage', 'EstimatedSalary', 'Portfolio Return', 'Diversification',
         'BusinessOwner', 'Revenue', 'LifeInsurance', 'NumTransactions', 'DaysSinceLastTransaction', 
         'ForeignAssets', 'NumProducts', 'Churn']]

df = df.sort_values('CustomerID')
df.to_csv('full_data.csv', index=False)
df.shape

(10000, 30)

### 4. train / test / validation

In [38]:
df['FLAG'] = 'FLAG'
for i in ['DaysSinceLastTransaction','Net Assets','HasCrCard','Married','ETF Health','Revenue','Diversification','ForeignAssets']:
    df[i+' FLAG'] = np.where(df[i].isnull(), 'A', 
                             np.where(df[i]<np.quantile(df[i],0.25), 'B',
                                      np.where(df[i]<np.quantile(df[i],0.50), 'C',
                                               np.where(df[i]<np.quantile(df[i],0.75), 'D', 'E'))))
    df['FLAG'] = df['FLAG']+'-'+df[i+' FLAG']
df.shape

(10000, 39)

In [39]:
train = pd.DataFrame()
test = pd.DataFrame()
val = pd.DataFrame()
for i in df['FLAG'].unique():
    gb = df[df['FLAG']==i].reset_index(drop=True).reset_index()
    train = pd.concat([train, gb[gb['index']%4==0]])
    train = pd.concat([train, gb[gb['index']%4==1]])
    test = pd.concat([test, gb[gb['index']%4==2]])
    val = pd.concat([val, gb[gb['index']%4==3]])

print(train.shape)
print(test.shape)
print(val.shape)
print(train.shape[0]+test.shape[0]+val.shape[0])

(5120, 40)
(2462, 40)
(2418, 40)
10000


### 5. export data

In [40]:
train = train.drop(['index', 'FLAG', 'DaysSinceLastTransaction FLAG', 'Net Assets FLAG', 'HasCrCard FLAG', 
                    'Married FLAG', 'ETF Health FLAG', 'Revenue FLAG', 'Diversification FLAG', 'ForeignAssets FLAG'], axis=1)
train = train.sort_values('CustomerID')
train.to_csv('train_data.csv', index=False)
print(train.shape)

test = test.drop(['index', 'FLAG', 'DaysSinceLastTransaction FLAG', 'Net Assets FLAG', 'HasCrCard FLAG', 
                    'Married FLAG', 'ETF Health FLAG', 'Revenue FLAG', 'Diversification FLAG', 'ForeignAssets FLAG'], axis=1)
test = test.sort_values('CustomerID')
test.to_csv('test_data.csv', index=False)
print(test.shape)

val = val.drop(['index', 'FLAG', 'DaysSinceLastTransaction FLAG', 'Net Assets FLAG', 'HasCrCard FLAG', 
                    'Married FLAG', 'ETF Health FLAG', 'Revenue FLAG', 'Diversification FLAG', 'ForeignAssets FLAG'], axis=1)
val = val.sort_values('CustomerID')
val.to_csv('val_data.csv', index=False)
print(val.shape)

(5120, 30)
(2462, 30)
(2418, 30)
